## $ \text{II} $ - Données relationnelles (Partie Cora)

In [ ]:
! pip install xgboost

In [ ]:
import pandas as  pd
import numpy as np
import scipy.io
import os
import altair as alt
from matplotlib import pyplot as plt
from sklearn.utils._testing import ignore_warnings
from sklearn.exceptions import ConvergenceWarning
from warnings import filterwarnings
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix, f1_score, make_scorer, recall_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier

# Étude exploratoire préliminaire

## Chargement des données

In [ ]:
# Chemin vers le dossier contenant les fichiers .mat
path = "/Users/moustaphakebe1998gmail.com/Documents/APPRENTISSAGE MACHINE/Projet_Apprentissage_supervisé-20241117/Données_relationnelles"

# Dictionnaire pour stocker les matrices pour chaque dataset
datasets = {}

# Charger les fichiers et extraire les matrices
for file_name in os.listdir(path):
    if file_name.endswith('.mat'):
        print(f"Chargement du fichier : {file_name}")

        # Charger les données .mat avec scipy
        mat_data = scipy.io.loadmat(os.path.join(path, file_name))

        # Nom du dataset (sans extension)
        dataset_name = os.path.splitext(file_name)[0]

        # Extraire les matrices importantes (fea, W, gnd)
        try:
            fea = pd.DataFrame(mat_data['fea'])  # Matrice X
            W = pd.DataFrame(mat_data['W'])     # Matrice d'adjacence
            gnd = pd.DataFrame(mat_data['gnd'], columns=['label'])  # Labels

            # Stocker les matrices dans le dictionnaire
            datasets[dataset_name] = {'fea': fea, 'W': W, 'gnd': gnd}

            print(f"Dataset '{dataset_name}' chargé avec succès !")
            print(f"Dimensions : fea={fea.shape}, W={W.shape}, gnd={gnd.shape}")
        except KeyError as e:
            print(f"Erreur : Clé manquante dans le fichier {file_name} ({e})")

Chargement du fichier : citeseer.mat
Dataset 'citeseer' chargé avec succès !
Dimensions : fea=(3327, 3703), W=(3327, 3327), gnd=(3327, 1)
Chargement du fichier : pubmed.mat
Dataset 'pubmed' chargé avec succès !
Dimensions : fea=(19717, 500), W=(19717, 1), gnd=(19717, 1)
Chargement du fichier : cora.mat
Dataset 'cora' chargé avec succès !
Dimensions : fea=(2708, 1433), W=(2708, 2708), gnd=(2708, 1)


## Exemple d'accés aux datasets

In [ ]:
cora_fea = datasets['cora']['fea']
cora_W = datasets['cora']['W']
cora_gnd = datasets['cora']['gnd']

print("Aperçu des données Cora :")
print(cora_fea.head(2))
print(cora_W.head(2))
print(cora_gnd.head(2))

Aperçu des données Cora :
   0     1     2     3     4     5     6     7     8     9     ...  1423  \
0     0     0     0     0     0     0     0     0     0     0  ...     0   
1     0     0     0     0     0     0     0     0     0     0  ...     0   

   1424  1425  1426  1427  1428  1429  1430  1431  1432  
0     0     0     0     0     0     0     0     0     0  
1     0     0     0     0     0     0     0     0     0  

[2 rows x 1433 columns]
   0     1     2     3     4     5     6     7     8     9     ...  2698  \
0     0     0     0     0     0     0     0     0     0     0  ...     0   
1     0     0     1     0     0     0     0     0     0     0  ...     0   

   2699  2700  2701  2702  2703  2704  2705  2706  2707  
0     0     0     0     0     0     0     0     0     0  
1     0     0     0     0     0     0     0     0     0  

[2 rows x 2708 columns]
   label
0      4
1      5


In [ ]:
citeseer_fea = datasets['citeseer']['fea']
citeseer_W = datasets['citeseer']['W']
citeseer_gnd = datasets['citeseer']['gnd']

print("Aperçu des données Citeseer :")
print(citeseer_fea.head(2))
print(citeseer_W.head(2))
print(citeseer_gnd.head(2))

Aperçu des données Citeseer :
   0     1     2     3     4     5     6     7     8     9     ...  3693  \
0     0     0     0     0     0     0     0     0     0     0  ...     0   
1     0     0     0     0     0     0     0     0     0     0  ...     0   

   3694  3695  3696  3697  3698  3699  3700  3701  3702  
0     0     0     0     0     0     0     0     0     0  
1     0     0     0     0     0     0     0     0     0  

[2 rows x 3703 columns]
   0     1     2     3     4     5     6     7     8     9     ...  3317  \
0     0     0     0     0     0     0     0     0     0     0  ...     0   
1     0     0     0     0     0     0     0     0     0     0  ...     0   

   3318  3319  3320  3321  3322  3323  3324  3325  3326  
0     0     0     0     0     0     0     0     0     0  
1     0     0     0     0     0     0     0     0     0  

[2 rows x 3327 columns]
   label
0      4
1      2


In [ ]:
pubmed_fea = datasets['pubmed']['fea']
pubmed_W = datasets['pubmed']['W']
pubmed_gnd = datasets['pubmed']['gnd']

print("Aperçu des données Pubmed :")
print(pubmed_fea.head(2))
print(pubmed_W.head(2))
print(pubmed_gnd.head(2))

Aperçu des données Pubmed :
   0    1    2    3    4    5    6         7    8    9    ...  490  491  492  \
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.004999  0.0  0.0  ...  0.0  0.0  0.0   
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.016434  0.0  0.0  ...  0.0  0.0  0.0   

   493  494  495  496  497  498  499  
0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[2 rows x 500 columns]
                                                   0
0    (0, 1378)\t1.0\n  (0, 1544)\t1.0\n  (0, 6092...
1    (0, 2943)\t1.0\n  (0, 8359)\t1.0\n  (0, 1019...
   label
0      2
1      2


In [ ]:
cora_W.head()

,0,1,2,3,4,5,6,7,8,9,...,2698,2699,2700,2701,2702,2703,2704,2705,2706,2707
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Analyse descriptive

In [ ]:
# Liste des noms des jeux de données
names=["pubmed", "cora", "citeseer"]
charts = []  # Pour stocker les graphiques
for name in names:
    # Compter les occurrences des classes
    stars_counts = datasets[name]['gnd'].value_counts().reset_index()
    stars_counts.columns = ['stars', 'count']

    # Création du graphique
    chart = alt.Chart(stars_counts).mark_bar().encode(
        x=alt.X('count:Q', title="Nombre d'individus"),
        y=alt.Y('stars:O', title='Classes', sort='-x'),
        color=alt.Color('stars:O', scale=alt.Scale(scheme='category20b'), legend=None),  # Couleur dépendant du label
        tooltip=['stars:O', 'count:Q']
    ).properties(
        title=f"Distribution des classes - {name.capitalize()}",
        width=300,
        height=200
    )

    # Ajouter le graphique à la liste
    charts.append(chart)

# Afficher les graphiques horizontalement
alt.hconcat(*charts).resolve_scale(color='independent')

alt.HConcatChart(...)


KeyboardInterrupt



## Calcul du nombre de voisins moyens

In [ ]:
charts = []
names=["cora", "citeseer"]
# Exemple avec 'cora', mais tu peux appliquer cela à d'autres jeux de données
for name in names:
    W = datasets[name]['W']  # Matrice d'adjacence du graphe

    # Calcul du degré des nœuds (somme des lignes de W)
    node_degrees = W.sum(axis=1)# W est une matrice creuse, donc on utilise .A pour la convertir en array dense
    # Calcul du nombre de voisins moyens
    avg_neighbors = np.mean(node_degrees)

    # 1. Visualisation de la distribution des degrés des nœuds
    degree_df = pd.DataFrame(node_degrees, columns=['degree'])

    degree_chart = alt.Chart(degree_df).mark_bar().encode(
        x=alt.X('degree:Q', bin=True, title='Degré des nœuds'),
        y=alt.Y('count():Q', title='Fréquence'),
        color=alt.Color('degree:Q', scale=alt.Scale(scheme='viridis'), legend=None),
        tooltip=['degree:Q', 'count():Q']
    ).properties(
        title=f"Distribution des degrés des nœuds - {name.capitalize()}",
        width=100,
        height=100
    )

    # 2. Affichage du nombre moyen de voisins
    avg_neighbors_chart = alt.Chart(pd.DataFrame({'avg_neighbors': [avg_neighbors]})).mark_text(
        align='center',
        baseline='middle',
        fontSize=20,
        fontWeight='bold',
        color='red'
    ).encode(
        x=alt.X('avg_neighbors:Q', title='Nombre moyen de voisins'),
        y=alt.Y('avg_neighbors:Q', title='Moyenne'),
        text=alt.Text('avg_neighbors:Q', format='.2f')
    ).properties(
        title=f"Nombre moyen de voisins - {name.capitalize()}",
        width=100,
        height=100
    )
    # Ajout des graphiques à la liste
    charts.append(alt.hconcat(degree_chart, avg_neighbors_chart))

# Affichage des graphiques pour tous les jeux de données
alt.vconcat(*charts)





alt.VConcatChart(...)

## Combinaison des informations de la matrice ajacente et la matrice initiale.
    Calcule la matrice combinée M = D^(-1) * W * X où :
    - D est la matrice diagonale des sommes des lignes de W
    - W est la matrice d'adjacence
    - X est la matrice des features

In [ ]:
#Je définis la fonction combinaison pour le calcul
def combination_matrix(W, X):
    """
    Calcule la matrice combinée M = D^(-1) * W * X où :
    - D est la matrice diagonale des sommes des lignes de W
    - W est la matrice d'adjacence
    - X est la matrice des features

    Args:
        W (np.ndarray): Matrice d'adjacence (n x n)
        X (np.ndarray): Matrice des features (n x d)

    Returns:
        np.ndarray: Matrice combinée M (n x d)
    """
    # Vérifier la compatibilité des dimensions
    if W.shape[0] != W.shape[1]:
        raise ValueError("La matrice W doit être carrée.")
    if W.shape[0] != X.shape[0]:
        raise ValueError("Le nombre de lignes de W doit correspondre au nombre de lignes de X.")

    # Calcul de la matrice D (diagonale des sommes des lignes de W)
    D = np.diag(W.sum(axis=1))

    # Vérifier si D est inversible (pas de zéros diagonaux)
    if np.any(np.diag(D) == 0):
        raise ValueError("La matrice D contient des zéros sur la diagonale, elle n'est pas inversible.")

    # Calcul de D^(-1)
    D_inv = np.linalg.inv(D)

    # Calcul de M
    M = D_inv @ W @ X

    return M

In [ ]:
M_citeseer=combination_matrix(citeseer_W, citeseer_fea)
M_cora=combination_matrix(cora_W, cora_fea)

In [ ]:
print(f'Dimension de la matrice Citeseer: {M_citeseer.shape}')
print(f'Dimension de la matrice Cora: {M_cora.shape}')

Dimension de la matrice Citeseer: (3327, 3703)
Dimension de la matrice Cora: (2708, 1433)


# CORA

In [ ]:
validation_size = 0.20
seed=42
X_coratrain, X_coravalidation, Y_coratrain, Y_coravalidation = train_test_split(M_cora, cora_gnd, test_size=validation_size, random_state=seed)

In [ ]:
filterwarnings("ignore", category=ConvergenceWarning)
# Validation croisée
num_folds = 10
seed = 7
# Définitions des métriques
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted'),
    'f1': make_scorer(f1_score, average='weighted')
}
pipelines=[]
pipelines.append(('ScaledLogisticR', Pipeline([('Scaler',StandardScaler()), ('LR', LogisticRegression())])))
pipelines.append(('ScaledLDA', Pipeline([('Scaler', StandardScaler()),('LDA', LinearDiscriminantAnalysis())])))
pipelines.append(('ScaledKNN', Pipeline([('Scaler', StandardScaler()),('KNN', KNeighborsClassifier(n_neighbors=4))])))
pipelines.append(('ScaledCART', Pipeline([('Scaler', StandardScaler()),('CART', DecisionTreeClassifier())])))
#pipelines.append(('ScaledNB', Pipeline([('Scaler', StandardScaler()),('NB', GaussianNB())])))
pipelines.append(('ScaledGB', Pipeline([('Scaler', StandardScaler()),('GB', GradientBoostingClassifier())])))
pipelines.append(('ScaledSVMlin', Pipeline([('Scaler', StandardScaler()),('SVM', SVC(kernel='linear'))])))
pipelines.append(('ScaledXGB', Pipeline([('Scaler', StandardScaler()), ('XGB', xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))])))
results = {metric: [] for metric in scoring.keys()}
names = []

for name, model in pipelines:
    kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)

    if name != 'ScaledXGB':
        cv_results = cross_validate(
            model,
            X_coratrain,
            Y_coratrain,
            cv=kfold,
            scoring=scoring,
            return_train_score=False
        )
        for metric in scoring.keys():
            results[metric].append(cv_results[f'test_{metric}'])
        names.append(name)
        print(f"{name}:")
        for metric in scoring.keys():
            print(f"  {metric.capitalize()} = {cv_results[f'test_{metric}'].mean():.4f} ± {cv_results[f'test_{metric}'].std():.4f}")
    else:
        # XGBoost: ajustement pour les classes
        # Comme XGBoost s'attend à ce que les i classes soient codées entre [0,..,i-1].
        Y_coratrain = Y_coratrain - 1
        Y_coravalidation = Y_coravalidation - 1
        cv_results = cross_validate(
            model,
            X_coratrain,
            Y_coratrain,
            cv=kfold,
            scoring=scoring,
            return_train_score=False
        )
        for metric in scoring.keys():
            results[metric].append(cv_results[f'test_{metric}'])
        names.append(name)
        print(f"{name}:")
        for metric in scoring.keys():
            print(f"  {metric.capitalize()} = {cv_results[f'test_{metric}'].mean():.4f} ± {cv_results[f'test_{metric}'].std():.4f}")

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLogisticR:
  Accuracy = 0.7904 ± 0.0189
  Precision = 0.8008 ± 0.0162
  Recall = 0.7904 ± 0.0189
  F1 = 0.7910 ± 0.0182


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLDA:
  Accuracy = 0.6874 ± 0.0172
  Precision = 0.7083 ± 0.0180
  Recall = 0.6874 ± 0.0172
  F1 = 0.6908 ± 0.0178


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected

...

ScaledGB:
  Accuracy = 0.8130 ± 0.0354
  Precision = 0.8242 ± 0.0306
  Recall = 0.8130 ± 0.0354
  F1 = 0.8121 ± 0.0357


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledSVMlin:
  Accuracy = 0.7775 ± 0.0199
  Precision = 0.7904 ± 0.0206
  Recall = 0.7775 ± 0.0199
  F1 = 0.7792 ± 0.0199


/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:35:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:35:36] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:35:40] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [21:35:43] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/App

ScaledXGB:
  Accuracy = 0.8223 ± 0.0218
  Precision = 0.8281 ± 0.0210
  Recall = 0.8223 ± 0.0218
  F1 = 0.8215 ± 0.0221


In [ ]:
! pip install prettytable

In [ ]:
from prettytable import PrettyTable
All_resultsCora = {
    "Model": ["ScaledLogisticR", "ScaledLDA", "ScaledKNN", "ScaledCART", "ScaledGB", "ScaledSVMlin", "ScaledXGB"],
    "Accuracy": [0.7904, 0.6874, 0.7258, 0.7073, 0.8130, 0.7775, 0.8223],
    "Precision": [0.8008, 0.7083, 0.7570, 0.7175, 0.8242, 0.7904, 0.8350],
    "Recall": [0.7904, 0.6874, 0.7258, 0.7073, 0.8130, 0.7775, 0.8223],
    "F1-Score": [0.7910, 0.6908, 0.7266, 0.7083, 0.8121, 0.7792, 0.8263],
}
# Création du DataFrame
df_results = pd.DataFrame(All_resultsCora)
# Création du tableau PrettyTable
table = PrettyTable()
table.field_names = ["Model", "Accuracy", "Precision", "Recall", "F1-Score"]
# Ajout des lignes
for _, row in df_results.iterrows():
    table.add_row(row)
print(table)

+-----------------+----------+-----------+--------+----------+
|      Model      | Accuracy | Precision | Recall | F1-Score |
+-----------------+----------+-----------+--------+----------+
| ScaledLogisticR |  0.7904  |   0.8008  | 0.7904 |  0.791   |
|    ScaledLDA    |  0.6874  |   0.7083  | 0.6874 |  0.6908  |
|    ScaledKNN    |  0.7258  |   0.757   | 0.7258 |  0.7266  |
|    ScaledCART   |  0.7073  |   0.7175  | 0.7073 |  0.7083  |
|     ScaledGB    |  0.813   |   0.8242  | 0.813  |  0.8121  |
|   ScaledSVMlin  |  0.7775  |   0.7904  | 0.7775 |  0.7792  |
|    ScaledXGB    |  0.8223  |   0.835   | 0.8223 |  0.8263  |
+-----------------+----------+-----------+--------+----------+


### Analyse générale

1. Gradient Boosting (GB) et XGBoost (XGB) sont nos meilleurs modèles, avec des métriques élevées et une faible variance. Cela les rend idéaux pour des données complexes et déséquilibrées.

2. Régression Logistique fournit des résultats robustes pour un modèle linéaire, ce qui peut être suffisant si la simplicité et l'interprétabilité sont vos priorités.

3. Les modèles CART, KNN, et LDA sont moins performants, mais peuvent être utiles si vous cherchez une simplicité supplémentaire ou si vos données sont modifiées.

### *Recommandations*

On peut utiliser XGBoost ou Gradient Boosting comme nos principaux modéles.Si on cherche une solution simple et interprétable, il est preferable d'utiliser la Régression Logistique.


### Visualisation des resultats

In [ ]:
# Préparer les données au format long
data = []
for metric, metric_results in results.items():
    for model_name, scores in zip(names, metric_results):
        for score in scores:
            data.append({'Metric': metric.capitalize(), 'Model': model_name, 'Score': score})

df_long = pd.DataFrame(data)

# Ajouter tooltips au graphique
chart = alt.Chart(df_long).mark_boxplot(extent='min-max').encode(
    x=alt.X('Model:N', title='Algorithm'),
    y=alt.Y('Score:Q', title='Score'),
    color='Model:N',
    tooltip=[
        alt.Tooltip('Model:N', title='Algorithm'),
        alt.Tooltip('Score:Q', title='Score', format='.4f'),
        alt.Tooltip('Metric:N', title='Metric')
    ],
    column=alt.Column('Metric:N', title='Metric', spacing=10)
).properties(
    title='Comparison of Algorithms for Multiple Metrics'
).configure_title(
    fontSize=20,
    anchor='start'
).configure_axis(
    labelFontSize=20,
    titleFontSize=20
)

chart

/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

Dans la suite de mon analyse je vais appliquer du sur-echantillonage , sous-echantillonage et la combinaison des deux méthodes pour voir le comportement des differents modeles

# Echantillonnage

In [ ]:
! pip install imblearn -q

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from collections import Counter

Je définis la fonction qui permet d'utiliser le type d'échantillonage à appliquer à nos données

In [ ]:
def evaluate_models(X, y, sampling_strategy="none"):
    """
    Applique oversampling ou undersampling sur les données, puis évalue plusieurs modèles
    via validation croisée.

    Parameters:
    - X: Features (DataFrame ou ndarray)
    - y: Target (Series ou ndarray)
    - sampling_strategy: "oversample" pour SMOTE, "undersample" pour RandomUnderSampler, Oversampling + Undersampling, ou "none" pour aucun équilibrage.

    Returns:
    - results: Dictionnaire des scores pour chaque métrique et chaque modèle.
    """
    # Application du sampling si spécifié
    if sampling_strategy == "oversample":
        smote = SMOTE(random_state=42)
        X, y = smote.fit_resample(X, y)
    elif sampling_strategy == "undersample":
        rus = RandomUnderSampler(random_state=42)
        X, y = rus.fit_resample(X, y)
    elif sampling_strategy== "Oversampling + Undersampling":
        smote_enn = SMOTEENN(random_state=42)
        X, y = smote_enn.fit_resample(X, y)

    # Validation croisée
    num_folds = 10
    seed = 7
    # Définitions des métriques
    scoring = {
        'accuracy': make_scorer(accuracy_score),
        'precision': make_scorer(precision_score, average='weighted'),
        'recall': make_scorer(recall_score, average='weighted'),
        'f1': make_scorer(f1_score, average='weighted')
    }
    pipelines=[]
    pipelines.append(('ScaledLogisticR', Pipeline([('Scaler',StandardScaler()), ('LR', LogisticRegression())])))
    pipelines.append(('ScaledLDA', Pipeline([('Scaler', StandardScaler()),('LDA', LinearDiscriminantAnalysis())])))
    pipelines.append(('ScaledKNN', Pipeline([('Scaler', StandardScaler()),('KNN', KNeighborsClassifier(n_neighbors=4))])))
    pipelines.append(('ScaledCART', Pipeline([('Scaler', StandardScaler()),('CART', DecisionTreeClassifier())])))
    #pipelines.append(('ScaledNB', Pipeline([('Scaler', StandardScaler()),('NB', GaussianNB())])))
    pipelines.append(('ScaledGB', Pipeline([('Scaler', StandardScaler()),('GB', GradientBoostingClassifier())])))
    pipelines.append(('ScaledSVMlin', Pipeline([('Scaler', StandardScaler()),('SVM', SVC(kernel='linear'))])))
    pipelines.append(('ScaledXGB', Pipeline([('Scaler', StandardScaler()), ('XGB', xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss'))])))
    results = {metric: [] for metric in scoring.keys()}
    names = []


    for name, model in pipelines:
        kfold = KFold(n_splits=num_folds, shuffle=True, random_state=seed)

        if name != 'ScaledXGB':
            cv_results = cross_validate(
                model,
                X,
                y,
                cv=kfold,
                scoring=scoring,
                return_train_score=False
            )
            for metric in scoring.keys():
                results[metric].append(cv_results[f'test_{metric}'])
            names.append(name)
            print(f"{name}:")
            for metric in scoring.keys():
                print(f"  {metric.capitalize()} = {cv_results[f'test_{metric}'].mean():.4f} ± {cv_results[f'test_{metric}'].std():.4f}")
        else:
            # XGBoost: ajustement pour les classes
            # Comme XGBoost s'attend à ce que les i classes soient codées entre [0,..,i-1].
            y = y - 1
            Y_coravalidation = y - 1
            cv_results = cross_validate(
                model,
                X,
                y,
                cv=kfold,
                scoring=scoring,
                return_train_score=False
            )
            for metric in scoring.keys():
                results[metric].append(cv_results[f'test_{metric}'])
            names.append(name)
            print(f"{name}:")
            for metric in scoring.keys():
                print(f"  {metric.capitalize()} = {cv_results[f'test_{metric}'].mean():.4f} ± {cv_results[f'test_{metric}'].std():.4f}")

    return results

## Oversampling (Sur-Echantillonnage)

In [ ]:
results_smote = evaluate_models(X_coratrain, Y_coratrain, sampling_strategy="oversample")

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules

ScaledLogisticR:
  Accuracy = 0.9346 ± 0.0067
  Precision = 0.9350 ± 0.0070
  Recall = 0.9346 ± 0.0067
  F1 = 0.9336 ± 0.0072


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLDA:
  Accuracy = 0.8966 ± 0.0135
  Precision = 0.8979 ± 0.0131
  Recall = 0.8966 ± 0.0135
  F1 = 0.8921 ± 0.0149


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected

...

ScaledGB:
  Accuracy = 0.9121 ± 0.0079
  Precision = 0.9149 ± 0.0089
  Recall = 0.9121 ± 0.0079
  F1 = 0.9125 ± 0.0083


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledSVMlin:
  Accuracy = 0.9266 ± 0.0080
  Precision = 0.9271 ± 0.0078
  Recall = 0.9266 ± 0.0080
  F1 = 0.9250 ± 0.0086


/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [13:44:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [13:44:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [13:45:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [13:45:45] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/App

ScaledXGB:
  Accuracy = 0.9248 ± 0.0077
  Precision = 0.9268 ± 0.0082
  Recall = 0.9248 ± 0.0077
  F1 = 0.9252 ± 0.0079


In [ ]:
# Préparer les données au format long
names = [
    'Logistic Regression',
    'LDA',
    'KNN',
    'Decision Tree',
    'Gradient Boosting',
    'SVM (Linear)',
    'XGBoost'
]
data = []
for metric, metric_results in results_smote.items():
    for model_name, scores in zip(names, metric_results):
        for score in scores:
            data.append({'Metric': metric.capitalize(), 'Model': model_name, 'Score': score})

df_long = pd.DataFrame(data)

# Ajouter tooltips au graphique
chart = alt.Chart(df_long).mark_boxplot(extent='min-max').encode(
    x=alt.X('Model:N', title='Algorithm'),
    y=alt.Y('Score:Q', title='Score'),
    color='Model:N',
    tooltip=[
        alt.Tooltip('Model:N', title='Algorithm'),
        alt.Tooltip('Score:Q', title='Score', format='.4f'),
        alt.Tooltip('Metric:N', title='Metric')
    ],
    column=alt.Column('Metric:N', title='Metric', spacing=10)
).properties(
    title='Comparison of Algorithms for Multiple Metrics'
).configure_title(
    fontSize=20,
    anchor='start'
).configure_axis(
    labelFontSize=20,
    titleFontSize=20
)

chart

/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [ ]:
# Les données des modèles
from prettytable import PrettyTable
results = [
    {"Model": "ScaledLogisticRegression", "Accuracy": "0.9346 ± 0.0067", "Precision": "0.9350 ± 0.0070", "Recall": "0.9346 ± 0.0067", "F1": "0.9336 ± 0.0072"},
    {"Model": "ScaledLDA", "Accuracy": "0.8966 ± 0.0135", "Precision": "0.8979 ± 0.0131", "Recall": "0.8966 ± 0.0135", "F1": "0.8921 ± 0.0149"},
    {"Model": "ScaledKNN", "Accuracy": "0.8679 ± 0.0154", "Precision": "0.8774 ± 0.0143", "Recall": "0.8679 ± 0.0154", "F1": "0.8596 ± 0.0170"},
    {"Model": "ScaledCART", "Accuracy": "0.8501 ± 0.0110", "Precision": "0.8533 ± 0.0106", "Recall": "0.8501 ± 0.0110", "F1": "0.8501 ± 0.0107"},
    {"Model": "ScaledSVM(Linear)", "Accuracy": "0.9266 ± 0.0080", "Precision": "0.9271 ± 0.0078", "Recall": "0.9266 ± 0.0080", "F1": "0.9250 ± 0.0086"},
    {"Model": "ScaledGB", "Accuracy": "0.9121 ± 0.0079", "Precision": "0.9149 ± 0.0089", "Recall": "0.9121 ± 0.0079", "F1": "0.9125 ± 0.0083"},
    {"Model": "ScaledXGB", "Accuracy": "0.9248 ± 0.0077", "Precision": "0.9268 ± 0.0082", "Recall": "0.9248 ± 0.0077", "F1": "0.9252 ± 0.0079"}
]

# Initialisation de la table
table = PrettyTable()
table.field_names = ["Model", "Accuracy", "Precision", "Recall", "F1"]

# Ajout des lignes à la table
for result in results:
    table.add_row([result["Model"], result["Accuracy"], result["Precision"], result["Recall"], result["F1"]])

# Affichage de la table
print(table)


+--------------------------+-----------------+-----------------+-----------------+-----------------+
|          Model           |     Accuracy    |    Precision    |      Recall     |        F1       |
+--------------------------+-----------------+-----------------+-----------------+-----------------+
| ScaledLogisticRegression | 0.9346 ± 0.0067 | 0.9350 ± 0.0070 | 0.9346 ± 0.0067 | 0.9336 ± 0.0072 |
|        ScaledLDA         | 0.8966 ± 0.0135 | 0.8979 ± 0.0131 | 0.8966 ± 0.0135 | 0.8921 ± 0.0149 |
|        ScaledKNN         | 0.8679 ± 0.0154 | 0.8774 ± 0.0143 | 0.8679 ± 0.0154 | 0.8596 ± 0.0170 |
|        ScaledCART        | 0.8501 ± 0.0110 | 0.8533 ± 0.0106 | 0.8501 ± 0.0110 | 0.8501 ± 0.0107 |
|    ScaledSVM(Linear)     | 0.9266 ± 0.0080 | 0.9271 ± 0.0078 | 0.9266 ± 0.0080 | 0.9250 ± 0.0086 |
|         ScaledGB         | 0.9121 ± 0.0079 | 0.9149 ± 0.0089 | 0.9121 ± 0.0079 | 0.9125 ± 0.0083 |
|        ScaledXGB         | 0.9248 ± 0.0077 | 0.9268 ± 0.0082 | 0.9248 ± 0.0077 | 0.9252 ±

Le sur-echantillonnage nous permet d'avoir une amélioration significative de nos differents metrics.

Cela s'explique par le fait que l'equilibrage des classe permet aux modéles de mieux apprendre les classes minoritaires évitant un biais vers les classes majoritaires.


## Undersampling

In [ ]:
results_under = evaluate_models(X_coratrain, Y_coratrain, sampling_strategy="undersample")

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLogisticR:
  Accuracy = 0.7838 ± 0.0344
  Precision = 0.7973 ± 0.0284
  Recall = 0.7838 ± 0.0344
  F1 = 0.7846 ± 0.0324


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLDA:
  Accuracy = 0.5010 ± 0.0569
  Precision = 0.5266 ± 0.0533
  Recall = 0.5010 ± 0.0569
  F1 = 0.4992 ± 0.0552


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected

...

ScaledGB:
  Accuracy = 0.7752 ± 0.0328
  Precision = 0.7986 ± 0.0323
  Recall = 0.7752 ± 0.0328
  F1 = 0.7784 ± 0.0325


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledSVMlin:
  Accuracy = 0.7724 ± 0.0328
  Precision = 0.7845 ± 0.0317
  Recall = 0.7724 ± 0.0328
  F1 = 0.7726 ± 0.0328


/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:13:24] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:13:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:13:36] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:13:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/App

ScaledXGB:
  Accuracy = 0.8019 ± 0.0415
  Precision = 0.8122 ± 0.0416
  Recall = 0.8019 ± 0.0415
  F1 = 0.8020 ± 0.0413


In [ ]:
# Préparer les données au format long
names = [
    'Logistic Regression',
    'LDA',
    'KNN',
    'Decision Tree',
    'Gradient Boosting',
    'SVM (Linear)',
    'XGBoost'
]
data = []
for metric, metric_results in results_under.items():
    for model_name, scores in zip(names, metric_results):
        for score in scores:
            data.append({'Metric': metric.capitalize(), 'Model': model_name, 'Score': score})

df_long = pd.DataFrame(data)

# Ajouter tooltips au graphique
chart = alt.Chart(df_long).mark_boxplot(extent='min-max').encode(
    x=alt.X('Model:N', title='Algorithm'),
    y=alt.Y('Score:Q', title='Score'),
    color='Model:N',
    tooltip=[
        alt.Tooltip('Model:N', title='Algorithm'),
        alt.Tooltip('Score:Q', title='Score', format='.4f'),
        alt.Tooltip('Metric:N', title='Metric')
    ],
    column=alt.Column('Metric:N', title='Metric', spacing=10)
).properties(
    title='Comparison of Algorithms for Multiple Metrics'
).configure_title(
    fontSize=20,
    anchor='start'
).configure_axis(
    labelFontSize=20,
    titleFontSize=20
)
chart

/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/Applications/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

In [ ]:
table = PrettyTable()
table.field_names = ["Model", "Accuracy", "Precision", "Recall", "F1"]

# Ajout des résultats pour chaque modèle
results = [
    ["ScaledLogisticR", "0.7838 ± 0.0344", "0.7973 ± 0.0284", "0.7838 ± 0.0344", "0.7846 ± 0.0324"],
    ["ScaledLDA", "0.5010 ± 0.0569", "0.5266 ± 0.0533", "0.5010 ± 0.0569", "0.4992 ± 0.0552"],
    ["ScaledKNN", "0.6038 ± 0.0467", "0.7155 ± 0.0258", "0.6038 ± 0.0467", "0.6069 ± 0.0452"],
    ["ScaledCART", "0.6943 ± 0.0515", "0.7171 ± 0.0491", "0.6943 ± 0.0515", "0.6975 ± 0.0514"],
    ["ScaledGB", "0.7752 ± 0.0328", "0.7986 ± 0.0323", "0.7752 ± 0.0328", "0.7784 ± 0.0325"],
    ["ScaledSVMlin", "0.7724 ± 0.0328", "0.7845 ± 0.0317", "0.7724 ± 0.0328", "0.7726 ± 0.0328"],
    ["ScaledXGB", "0.8019 ± 0.0415", "0.8122 ± 0.0416", "0.8019 ± 0.0415", "0.8020 ± 0.0413"]  # Compléter les valeurs restantes si nécessaire
]

for row in results:
    table.add_row(row)

# Affichage de la table
print(table)

+-----------------+-----------------+-----------------+-----------------+-----------------+
|      Model      |     Accuracy    |    Precision    |      Recall     |        F1       |
+-----------------+-----------------+-----------------+-----------------+-----------------+
| ScaledLogisticR | 0.7838 ± 0.0344 | 0.7973 ± 0.0284 | 0.7838 ± 0.0344 | 0.7846 ± 0.0324 |
|    ScaledLDA    | 0.5010 ± 0.0569 | 0.5266 ± 0.0533 | 0.5010 ± 0.0569 | 0.4992 ± 0.0552 |
|    ScaledKNN    | 0.6038 ± 0.0467 | 0.7155 ± 0.0258 | 0.6038 ± 0.0467 | 0.6069 ± 0.0452 |
|    ScaledCART   | 0.6943 ± 0.0515 | 0.7171 ± 0.0491 | 0.6943 ± 0.0515 | 0.6975 ± 0.0514 |
|     ScaledGB    | 0.7752 ± 0.0328 | 0.7986 ± 0.0323 | 0.7752 ± 0.0328 | 0.7784 ± 0.0325 |
|   ScaledSVMlin  | 0.7724 ± 0.0328 | 0.7845 ± 0.0317 | 0.7724 ± 0.0328 | 0.7726 ± 0.0328 |
|    ScaledXGB    | 0.8019 ± 0.0415 | 0.8122 ± 0.0416 | 0.8019 ± 0.0415 | 0.8020 ± 0.0413 |
+-----------------+-----------------+-----------------+-----------------+-------

## Ovesampling and Undersampling

In [ ]:
results_smoteen = evaluate_models(X_coratrain, Y_coratrain, sampling_strategy="Oversampling + Undersampling")

/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLogisticR:
  Accuracy = 0.9898 ± 0.0031
  Precision = 0.9899 ± 0.0031
  Recall = 0.9898 ± 0.0031
  F1 = 0.9897 ± 0.0031


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledLDA:
  Accuracy = 0.9240 ± 0.0184
  Precision = 0.9251 ± 0.0178
  Recall = 0.9240 ± 0.0184
  F1 = 0.9193 ± 0.0215


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/neighbors/_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected

...

ScaledGB:
  Accuracy = 0.9647 ± 0.0082
  Precision = 0.9655 ± 0.0082
  Recall = 0.9647 ± 0.0082
  F1 = 0.9647 ± 0.0083


/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Applications/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:1300: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Ple

ScaledSVMlin:
  Accuracy = 0.9900 ± 0.0040
  Precision = 0.9902 ± 0.0039
  Recall = 0.9900 ± 0.0040
  F1 = 0.9900 ± 0.0040


/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:11:46] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:12:09] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:12:34] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Applications/anaconda3/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [16:12:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/App

ScaledXGB:
  Accuracy = 0.9736 ± 0.0089
  Precision = 0.9744 ± 0.0085
  Recall = 0.9736 ± 0.0089
  F1 = 0.9737 ± 0.0088


In [ ]:
from prettytable import PrettyTable
table = PrettyTable()


table.field_names = ["Model", "Accuracy", "Precision", "Recall", "F1"]

# Ajout des résultats pour chaque modèle
results = [
    ["ScaledLogisticR", "0.9898 ± 0.0031", "0.9899 ± 0.0031", "0.9898 ± 0.0031", "0.9897 ± 0.0031"],
    ["ScaledLDA", "0.9240 ± 0.0184", "0.9251 ± 0.0178", "0.9240 ± 0.0184", "0.9193 ± 0.0215"],
    ["ScaledKNN", "0.9399 ± 0.0108", "0.9438 ± 0.0091", "0.9399 ± 0.0108", "0.9349 ± 0.0131"],
    ["ScaledCART", "0.9172 ± 0.0173", "0.9189 ± 0.0168", "0.9172 ± 0.0173", "0.9173 ± 0.0172"],
    ["ScaledGB", "0.9647 ± 0.0082", "0.9655 ± 0.0082", "0.9647 ± 0.0082", "00.9647 ± 0.0083"],
    ["ScaledSVMlin", "0.9900 ± 0.0040", "0.9902 ± 0.0039", "0.7724 ± 0.0328", "0.9900 ± 0.0040"],

    ["ScaledXGB", "0.9736 ± 0.0089", "0.9744 ± 0.0085", "0.9736 ± 0.0089", "0.9737 ± 0.0088"]  # Compléter les valeurs restantes si nécessaire
]

for row in results:
    table.add_row(row)

# Affichage de la table
print(table)

+-----------------+-----------------+-----------------+-----------------+------------------+
|      Model      |     Accuracy    |    Precision    |      Recall     |        F1        |
+-----------------+-----------------+-----------------+-----------------+------------------+
| ScaledLogisticR | 0.9898 ± 0.0031 | 0.9899 ± 0.0031 | 0.9898 ± 0.0031 | 0.9897 ± 0.0031  |
|    ScaledLDA    | 0.9240 ± 0.0184 | 0.9251 ± 0.0178 | 0.9240 ± 0.0184 | 0.9193 ± 0.0215  |
|    ScaledKNN    | 0.9399 ± 0.0108 | 0.9438 ± 0.0091 | 0.9399 ± 0.0108 | 0.9349 ± 0.0131  |
|    ScaledCART   | 0.9172 ± 0.0173 | 0.9189 ± 0.0168 | 0.9172 ± 0.0173 | 0.9173 ± 0.0172  |
|     ScaledGB    | 0.9647 ± 0.0082 | 0.9655 ± 0.0082 | 0.9647 ± 0.0082 | 00.9647 ± 0.0083 |
|   ScaledSVMlin  | 0.9900 ± 0.0040 | 0.9902 ± 0.0039 | 0.7724 ± 0.0328 | 0.9900 ± 0.0040  |
|    ScaledXGB    | 0.9736 ± 0.0089 | 0.9744 ± 0.0085 | 0.9736 ± 0.0089 | 0.9737 ± 0.0088  |
+-----------------+-----------------+-----------------+---------------